# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [8]:
!nvidia-smi || echo "CPU runtime"

/bin/bash: line 1: nvidia-smi: command not found
CPU runtime


In [38]:
# Installation des dépendances
!pip install -q --upgrade langchain langchain-community transformers sentencepiece accelerate langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 51.4 MB/s eta 0:00:00


## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [41]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline # Updated import for deprecated HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
# The 'langchain.chains' module was not found. We will use the Runnable interface instead.
# from langchain.chains import LLMChain

In [42]:

# TODO: choose a small model compatible with text-generation
model_name = "gpt2"  # Changed to gpt2 for text-generation compatibility


In [43]:

# TODO: load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name) # Changed to AutoModelForCausalLM


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [44]:

# TODO: create a generation pipeline
gen_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)


In [45]:

# TODO: build prompt + LLMChain for friendly rewriting
template = "Rewrite this text to be simpler for beginners:{text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
# Using the Runnable interface instead of LLMChain
chain = prompt | llm

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
# Using invoke for the Runnable chain
rewritten = chain.invoke({"text": sample_text})
print(rewritten)


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rewrite this text to be simpler for beginners:LangChain helps you build LLM apps by composing prompts, models, and tools. It is a powerful, powerful, and incredibly easy to read tool that can help you quickly and easily understand your code.You can learn more about the language and what it means by reading this series.


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [46]:
from langchain_core.prompts import PromptTemplate

#To-Do define run templates
summary_prompt = PromptTemplate(
    template="Summarize the following paragraph in a concise way:\n\n{paragraph}\n\nSummary:",
    input_variables=["paragraph"],
)
bullets_prompt = PromptTemplate(
    template="Convert the following summary into 3 bullet points:\n\n{summary}\n\nBullet points:",
    input_variables=["summary"],
)

In [47]:
# First stage: paragraph -> summary (string)
summary_chain = summary_prompt | llm

# Full chain:
# 1. Take input {"paragraph": ...}
# 2. Run summary_chain to get a summary string
# 3. Wrap into {"summary": summary}
# 4. Run bullets_prompt, then llm
summarize_then_bullets = (
    {"summary": summary_chain}   # this creates a dict runnable
    | bullets_prompt
    | llm
)

In [48]:
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Convert the following summary into 3 bullet points:

Summarize the following paragraph in a concise way:

LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows.

Summary:

LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows. LANG is an extension of the Language Toolkit (LWP).

What works for Django

First of all, if you run into any issues with LangChain, please add them to the review queue on the Django User Repository.

If you have any other issues with LangChain, your Django Repository, or Django is hosting any of the services you need, please contact us.

Please note:

This code

Bullet points:

LangChain supports an API with several different languages, which can be used for writing and maintaining applications.

LangChain is designed to work with D

## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [57]:
!pip install --upgrade langchain langchain-community

In [59]:
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory # Attempting import as requested by user

# Define a prompt template that incorporates conversation history
conversation_template = """The following is a friendly conversation between a human and an AI.

Current conversation:
{history}
Human: {input}
AI:"""
conversation_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=conversation_template
)

# NOTE: ConversationBufferMemory is expected here, but for testing purposes, we're changing the import.
# The rest of the function `conversational_chain_predict` and its usage will likely fail if ChatMessageHistory is not directly compatible.
memory = ChatMessageHistory() # Using ChatMessageHistory as per user request for testing

# A custom function to mimic convo.predict() by manually handling memory and LLM invocation.
# This is necessary because HuggingFacePipeline (our 'llm') expects a single string input.
def conversational_chain_predict(user_input: str, current_memory, llm_model, prompt_template: PromptTemplate):
    # Load history from memory
    # ChatMessageHistory stores messages as a list of Message objects.
    # This function expects a string history, so we'll need to adapt.
    # For this test, we'll temporarily represent history as a simple string based on existing messages.
    # In a real scenario with ChatMessageHistory, you would typically use a MessagesPlaceholder in your prompt
    # and pass the list of messages directly.
    current_history_str = "\n".join([f"{m.type.capitalize()}: {m.content}" for m in current_memory.messages])

    # Format the prompt with current history and new input
    full_prompt_string = prompt_template.format(history=current_history_str, input=user_input)

    # Invoke the LLM
    ai_response = llm_model.invoke(full_prompt_string)

    # Save the current interaction to memory
    # For ChatMessageHistory, we add messages directly
    current_memory.add_user_message(user_input)
    current_memory.add_ai_message(ai_response)

    return ai_response

# First turn
human_input1 = "Hi there! What's LangChain?"
reply1 = conversational_chain_predict(human_input1, memory, llm, conversation_prompt)
print("Turn 1:", reply1)

# Second turn
human_input2 = "Can it help me build a simple chatbot?"
reply2 = conversational_chain_predict(human_input2, memory, llm, conversation_prompt)
print("Turn 2:", reply2)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 1: The following is a friendly conversation between a human and an AI.

Current conversation:

Human: Hi there! What's LangChain?
AI: You're a big fan of the original Star Trek. Why the hell not?

Human: Well, I've been playing the first Star Trek game.

AI: Why the hell not?

Human: I'm not sure what to think, but I can tell you that you want to play Star Trek.

AI: Why the hell not?

Human: I'm not sure what to think, but I can tell you that you want to play Star Trek.

Human: I'm not sure what to think, but I can tell you that you want to play Star Trek.

Human
Turn 2: The following is a friendly conversation between a human and an AI.

Current conversation:
Human: Hi there! What's LangChain?
Ai: The following is a friendly conversation between a human and an AI.

Current conversation:

Human: Hi there! What's LangChain?
AI: You're a big fan of the original Star Trek. Why the hell not?

Human: Well, I've been playing the first Star Trek game.

AI: Why the hell not?

Human: I'm 

## Your observations (fill in)
- **Latency**: Relatively fast due to small model size and CPU runtime, but not explicitly measured.
- **Quality**: Low. The `gpt2` model produced highly repetitive and nonsensical text in both the rewriting, two-step pipeline, and conversational chain tasks.
- **Quirks**: Significant hallucinations and lack of coherence. The model often repeated parts of the prompt or previous turns within its responses, and did not maintain context effectively in the conversational chain. Warnings about `max_new_tokens` taking precedence over `max_length` were also observed.